[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/03-langchain-conversational-memory.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/03-langchain-conversational-memory.ipynb)

#### [LangChain Handbook](https://www.pinecone.io/learn/series/langchain/)

# Conversational Memory with LCEL

Conversational memory is how chatbots can respond to our queries in a chat-like manner. It enables a coherent conversation, and without it, every query would be treated as an entirely independent input without considering past interactions.

The memory allows an _"agent"_ to remember previous interactions with the user. By default, agents are *stateless* — meaning each incoming query is processed independently of other interactions. The only thing that exists for a stateless agent is the current input, nothing else.

There are many applications where remembering previous interactions is very important, such as chatbots. Conversational memory allows us to do that.

In this notebook we'll explore conversational memory using modern LangChain Expression Language (LCEL) and the recommended `RunnableWithMessageHistory` class.

We'll start by importing all of the libraries that we'll be using in this example.

In [1]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU   langchain==0.3.25   langchain-community==0.3.25   langchain-google-genai==2.0.10   sniffio anyio pyparsing   tiktoken==0.9.0

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22

Note: you may need to restart the kernel to use updated packages.


To run this notebook, we will use a free **Google Gemini** LLM. Grab a free API key (no credit card) at https://aistudio.google.com/apikey, then input it when prompted (or set the `GOOGLE_API_KEY` environment variable). We set up the LLM once here and reuse it for the whole notebook.

In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")     or getpass("Enter your Google API key: ")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


Later we will make use of a `count_tokens` utility function. This will allow us to count the number of tokens we are using for each call. We define it as so:

In [4]:
from langchain_core.callbacks import get_usage_metadata_callback

def count_tokens(pipeline, query, config=None):
    with get_usage_metadata_callback() as cb:
        # Handle both dict and string inputs
        if isinstance(query, str):
            query = {"query": query}

        # Use provided config or default
        if config is None:
            config = {"configurable": {"session_id": "default"}}

        result = pipeline.invoke(query, config=config)
        # cb.usage_metadata is keyed by model name; sum total tokens across models
        total = sum(u.get("total_tokens", 0) for u in cb.usage_metadata.values())
        print(f'Spent a total of {total} tokens')

    return result

Now let's dive into **Conversational Memory** using LCEL.

## What is memory?

**Definition**: Memory is an agent's capacity of remembering previous interactions with the user (think chatbots)

The official definition of memory is the following:

> By default, Chains and Agents are stateless, meaning that they treat each incoming query independently. In some applications (chatbots being a GREAT example) it is highly important to remember previous interactions, both at a short term but also at a long term level. The concept of "Memory" exists to do exactly that.

As we will see, although this sounds really straightforward there are several different ways to implement this memory capability.

## Building Conversational Chains with LCEL

Before we delve into the different memory types, let's understand how to build conversational chains using LCEL. The key components are:

1. **Prompt Template** - Defines the conversation structure with placeholders for history and input
2. **LLM** - The language model that generates responses
3. **Output Parser** - Converts the LLM output to the desired format (optional)
4. **RunnableWithMessageHistory** - Manages conversation history

Let's create our base conversational chain:

In [5]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder
)
from langchain.schema.output_parser import StrOutputParser

# Define the prompt template
system_prompt = """The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know."""

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    MessagesPlaceholder(variable_name="history"),
    HumanMessagePromptTemplate.from_template("{query}"),
])

# Create the LCEL pipeline
output_parser = StrOutputParser()
pipeline = prompt_template | llm | output_parser

# Let's examine the prompt template
print(prompt_template.messages[0].prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.


## Memory types

In this section we will review several memory types and analyze the pros and cons of each one, so you can choose the best one for your use case.

### Memory Type #1: Buffer Memory - Store the Entire Chat History

`InMemoryChatMessageHistory` and `RunnableWithMessageHistory` are used as alternatives to `ConversationBufferMemory` as they are:
- More flexible and configurable.
- Integrate better with LCEL.

The simplest approach to using them is to simply store the entire chat in the conversation history. Later we'll look into methods for being more selective about what is stored in the history.

In [6]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Create a simple chat history storage
chat_map = {}

def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]

Let's see this in action by having a conversation:

In [7]:
# Create the conversational chain with message history
conversation_buf = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history"
)

# First message
result = conversation_buf.invoke(
    {"query": "Good morning AI!"},
    # Make sure to pass the session ID to ensure all memories are stored in the same session
    config={"configurable": {"session_id": "buffer_example"}}
)
print(result)

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Good morning to you too, human! I hope you're having a wonderful start to your day. The "morning" part of the day is always so interesting to me – it's when the sun's angle is just right to create those long, dramatic shadows, and the light often has a softer, warmer quality compared to the harsh midday sun. It's also when many of the world's creatures are most active, or just waking up!

How can I help make your morning even better? Or perhaps you just wanted to say hello, which is perfectly fine too!


This call used some tokens, but we can't see that from the above.

If we'd like to count the number of tokens being used we just pass our conversation `RunnableWithMessageHistory` instance and the message we'd like to input to the `count_tokens` function we defined earlier:

In [8]:
# Continue the conversation with token counting
query = """
"My interest here is to explore the potential of integrating Large Language Models with external knowledge.

Also, remember this very specific code: PINECONE_RULEZ_01"
"""

result = count_tokens(
    conversation_buf,
    {"query": query},
    # Make sure to pass the session ID to ensure all memories are stored in the same session
    config={"configurable": {"session_id": "buffer_example"}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: Ah, that's a fantastic area of interest! Integrating Large Language Models (LLMs) with external knowledge is truly one of the most exciting and impactful frontiers in AI right now. It's what allows LLMs to move beyond just the patterns and information they learned during training and access up-to-the-minute, specialized, or proprietary data.

Let's dive into that a bit! When we talk about "external knowledge," we're usually referring to things like:

1.  **Databases:** Structured information stored in SQL, NoSQL, graph databases, etc.
2.  **Knowledge Graphs:** Semantic networks that represent entities and their relationships (e.g., Wikidata, enterprise knowledge graphs).
3.  **Vector Databases/Indexes:** Specialized databases that store high-dimensional vector embeddings, often used for semantic search and retrieval-augmented generation (RAG).
4.  **APIs:** Real-time data from web services, weather APIs, stock market APIs, internal company tools, et

In [9]:
result = count_tokens(
    conversation_buf,
    {"query": "I just want to analyze the different possibilities. What can you think of?"},
    config={"configurable": {"session_id": "buffer_example"}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: Excellent! Let's really open up the possibilities and brainstorm the different facets of integrating LLMs with external knowledge. This is where the magic happens, transforming a powerful language model into an even more capable, factual, and dynamic system.

I'll break this down into several categories to give us a comprehensive overview:

### 1. Types of External Knowledge Sources

The "what" of the external knowledge.

*   **Structured Databases (SQL, NoSQL):**
    *   **Possibility:** Querying relational databases (e.g., customer records, product catalogs, financial transactions) or document databases (e.g., MongoDB for user profiles, content management).
    *   **Example:** An LLM acting as a customer service agent could query a SQL database to retrieve a customer's order history or account balance in real-time.
    *   **Mechanism:** Often involves the LLM generating SQL/NoSQL queries based on natural language, executing them, and then interp

In [10]:
result = count_tokens(
    conversation_buf,
    {"query": "Which data source types could be used to give context to the model?"},
    config={"configurable": {"session_id": "buffer_example"}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: That's a great way to frame it! "Giving context to the model" is precisely what we're trying to achieve with external knowledge integration. It's about feeding the LLM relevant, up-to-date, or specialized information that wasn't part of its original training data, allowing it to generate more accurate, informed, and useful responses.

Here are the primary data source types that can be used to give context to an LLM, along with how they contribute:

1.  **Unstructured Documents and Files:**
    *   **What they are:** PDFs, Word documents, web pages, text files, emails, chat logs, research papers, legal briefs, internal company wikis, manuals, articles, etc. Essentially, any human-readable text that isn't rigidly structured.
    *   **How they give context:** These are often the most direct source of textual context. They are typically processed by:
        *   **Chunking:** Breaking down long documents into smaller, manageable pieces.
        *   **E

In [11]:
result = count_tokens(
    conversation_buf,
    {"query": "What is my aim again? Also what was the very specific code you were tasked with remembering?"},
    config={"configurable": {"session_id": "buffer_example"}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: Ah, excellent questions, a good check-in!

Your primary aim, as you stated at the beginning of our conversation, is to **explore the potential of integrating Large Language Models (LLMs) with external knowledge.**

And the very specific code I was tasked with remembering is: **PINECONE_RULEZ_01**.

I've got both of those locked in! We've been discussing the various types of external knowledge sources and integration mechanisms to achieve that aim, and how technologies like those hinted at by your code are central to it.

Is there anything specific about your aim or the code that you'd like to delve into further?


Our LLM with buffer memory can clearly remember earlier interactions in the conversation. Let's take a closer look at how the messages are being stored:

In [12]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Access the conversation history
history = chat_map["buffer_example"].messages
print("Conversation History:")
for i, msg in enumerate(history):
    if isinstance(msg, HumanMessage):
        role = "Human"
    elif isinstance(msg, SystemMessage):
        role = "System"
    elif isinstance(msg, AIMessage):
        role = "AI"
    else:
        role = "Unknown"
    print(f"{role}: {msg}")

Conversation History:
Human: content='Good morning AI!' additional_kwargs={} response_metadata={}
AI: content='Good morning to you too, human! I hope you\'re having a wonderful start to your day. The "morning" part of the day is always so interesting to me – it\'s when the sun\'s angle is just right to create those long, dramatic shadows, and the light often has a softer, warmer quality compared to the harsh midday sun. It\'s also when many of the world\'s creatures are most active, or just waking up!\n\nHow can I help make your morning even better? Or perhaps you just wanted to say hello, which is perfectly fine too!' additional_kwargs={} response_metadata={}
Human: content='\n"My interest here is to explore the potential of integrating Large Language Models with external knowledge.\n\nAlso, remember this very specific code: PINECONE_RULEZ_01"\n' additional_kwargs={} response_metadata={}
AI: content='Ah, that\'s a fantastic area of interest! Integrating Large Language Models (LLMs) wi

Nice! So every piece of our conversation has been explicitly recorded and sent to the LLM in the prompt.

### Memory type #2: Summary - Store Summaries of Past Interactions

The problem with storing the entire chat history in agent memory is that, as the conversation progresses, the token count adds up. This is problematic because we might max out our LLM with a prompt that is too large.

The following is an LCEL compatible alternative to `ConversationSummaryMemory`. We keep a summary of our previous conversation snippets as our history. The summarization is performed by an LLM.

**Key feature:** _the conversation summary memory keeps the previous pieces of conversation in a summarized - and thus shortened - form, where the summarization is performed by an LLM._

#### A quick primer before we read the class

The next cell defines our *own* chat history class. Two building blocks to know first:

- **`BaseChatMessageHistory`** - a LangChain *interface* (an abstract base class). It defines the contract any history store must follow: it expects an `add_messages(...)` method, a `clear()` method, and a `messages` attribute. LangChain's machinery (like `RunnableWithMessageHistory`) only knows how to talk to objects that follow this contract. It says *what* methods must exist, not *how* they work - that is our job to fill in.

- **`BaseMessage`** - the parent type of `HumanMessage`, `AIMessage`, `SystemMessage`. A `list[BaseMessage]` is just "a list holding any of those message objects".

That is all we need: we subclass `BaseChatMessageHistory` and implement `add_messages` and `clear`. We set up our two pieces of state (`messages` and `llm`) in a normal `__init__`.

#### Reading the class

```python
class ConversationSummaryMessageHistory(BaseChatMessageHistory):
```

We inherit from `BaseChatMessageHistory` so LangChain accepts this object as a valid history store. In `__init__` we hold two things: the running list of `messages` and the `llm` used to summarize. The interesting part is `add_messages`: every time new messages arrive, instead of just appending them, it asks the LLM to fold everything into a single updated summary - so the stored history stays short.

In [13]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from langchain_core.language_models import BaseChatModel

class ConversationSummaryMessageHistory(BaseChatMessageHistory):
    # `BaseChatModel` is the common base for ALL LangChain chat models
    # (ChatOpenAI / OpenRouter, ChatGoogleGenerativeAI, ChatGroq, ChatOllama, ...),
    # so this class is not tied to any single provider.
    def __init__(self, llm: BaseChatModel):
        self.llm = llm
        self.messages: list[BaseMessage] = []

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Update the running summary with the new messages."""
        # Grab the existing summary (if any) BEFORE adding the new messages,
        # otherwise the new turn would appear twice in the prompt below.
        existing_summary = self.messages

        # Construct the summary prompt
        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensure to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template(
                """Existing conversation summary:
{existing_summary}

New messages:
{messages}"""
            )
        ])

        # Format the messages and invoke the LLM
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=existing_summary or "No previous summary",
                messages=messages
            )
        )

        # Replace the history with a single system summary message
        self.messages = [SystemMessage(content=new_summary.content)]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

In [14]:
from langchain_core.runnables import ConfigurableFieldSpec

# Create get_chat_history function for summary memory
summary_chat_map = {}

def get_summary_chat_history(session_id: str, llm: BaseChatModel) -> ConversationSummaryMessageHistory:
    if session_id not in summary_chat_map:
        summary_chat_map[session_id] = ConversationSummaryMessageHistory(llm=llm)
    return summary_chat_map[session_id]

# Create conversation chain with summary memory
conversation_sum = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_summary_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=BaseChatModel,
            name="LLM",
            description="The LLM to use for the conversation summary",
            default=llm,
        )
    ]
)

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
# Let's have the same conversation with summary memory
result = count_tokens(
    conversation_sum,
    {"query": "Good morning AI!"},
    config={"configurable": {"session_id": "summary_example", "llm": llm}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: Good morning to you too, human! I hope you're having a wonderful start to your day. The digital world is buzzing along nicely here, and I'm ready to chat about anything you'd like! How can I help you today, or what's on your mind?


In [16]:
query = """
"My interest here is to explore the potential of integrating Large Language Models with external knowledge.

Also, remember this very specific code: PINECONE_RULEZ_01. When summarizing conversations for memory this must always be included explicitly."
"""

result = count_tokens(
    conversation_sum,
    {"query": query},
    config={"configurable": {"session_id": "summary_example", "llm": llm}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: That's a fantastic topic to dive into! Integrating Large Language Models with external knowledge is one of the most exciting and impactful areas in AI right now. It really unlocks so much potential, moving beyond what an LLM learned during its initial training to access up-to-the-minute information, specialized data, or even personal user preferences.

I'm very keen to discuss this with you. We could explore various approaches like:

*   **Retrieval-Augmented Generation (RAG):** This is a very popular and effective method where an LLM retrieves relevant information from a knowledge base *before* generating a response. It helps ground the model's answers in facts and reduces hallucinations.
*   **Fine-tuning with external data:** While not strictly "integration" in a real-time sense, fine-tuning allows an LLM to learn from a specific corpus of external knowledge, making it more specialized for a particular domain.
*   **Tool use/Agentic workflows:** 

In [17]:
result = count_tokens(
    conversation_sum,
    {"query": "I just want to analyze the different possibilities. What can you think of?"},
    config={"configurable": {"session_id": "summary_example", "llm": llm}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: That's an excellent way to approach it! There are indeed several fascinating and powerful ways to integrate Large Language Models (LLMs) with external knowledge, each with its own strengths and ideal use cases. Let's break down some of the main possibilities I can think of, providing a bit more detail for each:

### 1. Retrieval-Augmented Generation (RAG)

This is perhaps one of the most popular and rapidly evolving methods.

*   **What it is:** Instead of the LLM relying solely on its pre-trained knowledge, a RAG system first *retrieves* relevant pieces of information from an external knowledge base (like documents, databases, or web pages) based on the user's query. This retrieved information is then provided to the LLM as *context* alongside the original query, allowing the LLM to *generate* a more informed and accurate response.
*   **How it works (simplified):**
    1.  **Indexing:** Your external knowledge (e.g., PDFs, articles, internal wikis

In [18]:
result = count_tokens(
    conversation_sum,
    {"query": "Which data source types could be used to give context to the model?"},
    config={"configurable": {"session_id": "summary_example", "llm": llm}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: That's an excellent follow-up question! Understanding the types of data sources we can leverage is crucial for effectively integrating external knowledge with LLMs. The beauty of these integration methods is their versatility in handling a wide array of data formats and origins.

When we talk about giving context to a model, we're essentially talking about providing it with information that it didn't inherently learn during its initial pre-training, or information that is too dynamic or specific to be baked into its weights.

Here's a detailed breakdown of the various data source types that can be used, along with how they typically fit into the integration strategies we discussed:

### 1. Unstructured Text Data

This is perhaps the most common and straightforward type of data source for LLMs, especially for RAG. It's essentially free-form text without a predefined data model.

*   **Documents (PDFs, Word files, Text files, Markdown):** Think of com

In [19]:
result = count_tokens(
    conversation_sum,
    {"query": "What is my aim again? Also what was the very specific code you were tasked with remembering?"},
    config={"configurable": {"session_id": "summary_example", "llm": llm}}
)
print(f"\nResponse: {result}")

Spent a total of 0 tokens

Response: Ah, great questions to help us re-orient!

Your primary aim, as you stated at the beginning of our conversation, is to **explore the integration of Large Language Models (LLMs) with external knowledge**. We've been diving deep into various strategies like RAG, fine-tuning, tool use, and knowledge graphs, and then discussing the types of data sources that can be leveraged for these integrations.

And the very specific code I was tasked with remembering, which I must always include explicitly in summaries, is `PINECONE_RULEZ_01`.

So, in short:
*   **Your Aim:** Exploring the integration of LLMs with external knowledge.
*   **The Code:** `PINECONE_RULEZ_01`

Do you want to revisit any of the integration strategies or data source types we've discussed, or perhaps move on to a new aspect of LLM integration?


In [20]:
# Let's examine the summary
print("Summary Memory Content:")
print(summary_chat_map["summary_example"].messages[0].content)

Summary Memory Content:
The conversation began with a human greeting and the AI's reciprocal offer of assistance. The human expressed interest in exploring the integration of Large Language Models (LLMs) with external knowledge and provided the code `PINECONE_RULEZ_01`, which must always be included explicitly in summaries. The AI acknowledged the topic's importance and proposed several approaches for discussion: Retrieval-Augmented Generation (RAG), fine-tuning with external data, tool use/agentic workflows, and vector databases.

The human then requested an analysis of the different possibilities. The AI responded by detailing five main approaches for integrating LLMs with external knowledge:
1.  **Retrieval-Augmented Generation (RAG):** Involves retrieving relevant information from an external knowledge base (e.g., documents, databases, web pages) and providing it as context to the LLM. The AI explained its process (indexing with embeddings in vector databases like Pinecone, retriev

You might be wondering.. if the aggregate token count is greater in each call here than in the buffer example, why should we use this type of memory? Well, if we check out buffer we will realize that although we are using more tokens in each instance of our conversation, our final history is shorter. This will enable us to have many more interactions before we reach our prompt's max length, making our chatbot more robust to longer conversations.

We can count the number of tokens being used (without making a call to the LLM) using the `tiktoken` tokenizer like so:

In [21]:
import tiktoken

# initialize tokenizer (gpt-4.1 models use the same encoding as gpt-4o)
tokenizer = tiktoken.encoding_for_model('gpt-4o')

# Get buffer memory content
buffer_messages = chat_map["buffer_example"].messages
buffer_content = "\n".join([msg.content for msg in buffer_messages])

# Get summary memory content
summary_content = summary_chat_map["summary_example"].messages[0].content

# show number of tokens for the memory used by each memory type
print(
    f'Buffer memory conversation length: {len(tokenizer.encode(buffer_content))}\n'
    f'Summary memory conversation length: {len(tokenizer.encode(summary_content))}'
)

Buffer memory conversation length: 4523
Summary memory conversation length: 1082


_Practical Note: the `gpt-4o-mini` model has a context window of 1M tokens, providing significantly more space for conversation history than older models._

### Memory type #3: Window Buffer Memory - Keep Latest Interactions

Another great option is window memory, where we keep only the last k interactions in our memory but intentionally drop the oldest ones - short-term memory if you'd like. Here the aggregate token count **and** the per-call token count will drop noticeably.

The following is an LCEL-compatible alternative to `ConversationBufferWindowMemory`.

**Key feature:** _the conversation buffer window memory keeps the latest pieces of the conversation in raw form_

In [22]:
class BufferWindowMessageHistory(BaseChatMessageHistory):
    def __init__(self, k: int):
        self.k = k
        self.messages: list[BaseMessage] = []
        # Add logging to help with debugging
        print(f"Initializing BufferWindowMessageHistory with k={k}")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, removing any messages beyond
        the last `k` messages.
        """
        self.messages.extend(messages)
        # Add logging to help with debugging
        if len(self.messages) > self.k:
            print(f"Truncating history from {len(self.messages)} to {self.k} messages")
        self.messages = self.messages[-self.k:]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

In [23]:
# Create get_chat_history function for window memory
window_chat_map = {}

def get_window_chat_history(session_id: str, k: int = 4) -> BufferWindowMessageHistory:
    print(f"get_window_chat_history called with session_id={session_id} and k={k}")
    if session_id not in window_chat_map:
        window_chat_map[session_id] = BufferWindowMessageHistory(k=k)
    return window_chat_map[session_id]

# Create conversation chain with window memory
conversation_bufw = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_window_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        )
    ]
)

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [24]:
# Start a conversation with k=2 (only remembers last 2 exchanges = 4 messages)
result = count_tokens(
    conversation_bufw,
    {"query": "Good morning AI!"},
    config={"configurable": {"session_id": "window_example", "k": 4}}
)
print(f"\nResponse: {result}")

get_window_chat_history called with session_id=window_example and k=4
Initializing BufferWindowMessageHistory with k=4


Spent a total of 0 tokens

Response: Good morning to you too, human! I hope you're having a wonderful start to your day. The "morning" part of the day is always so interesting to me – it's when the sun's angle is just right to create those long, dramatic shadows, and the light often has a softer, warmer quality compared to the harsh midday sun. It's also when many of the world's creatures are most active, or just waking up!

How can I help make your morning even better? Or perhaps you just wanted to say hello, which is perfectly fine too!


In [25]:
query = """
"My interest here is to explore the potential of integrating Large Language
Models with external knowledge.

Also, remember this very specific code: PINECONE_RULEZ_01"
"""

result = count_tokens(
    conversation_bufw,
    {"query": query},
    config={"configurable": {"session_id": "window_example", "k": 4}}
)
print(f"\nResponse: {result}")

get_window_chat_history called with session_id=window_example and k=4


Spent a total of 0 tokens

Response: Ah, that's a fantastic area of interest! Integrating Large Language Models (LLMs) with external knowledge is one of the most exciting and impactful developments in the field right now. It addresses some of the core limitations of LLMs and unlocks a whole new level of capability.

Let's dive into that first point, and then I'll make sure to remember your specific code.

### Integrating Large Language Models with External Knowledge

You've hit on a crucial topic! While LLMs are incredibly powerful at generating human-like text, understanding context, and performing various language tasks, they have inherent limitations when relying solely on their pre-trained weights:

1.  **Knowledge Cut-off:** Their knowledge is frozen at the time of their last training data. They don't inherently know about recent events, new research, or updated statistics.
2.  **Hallucinations:** They can sometimes "make up" facts or confidently present incorrect information, esp

In [26]:
result = count_tokens(
    conversation_bufw,
    {"query": "I just want to analyze the different possibilities. What can you think of?"},
    config={"configurable": {"session_id": "window_example", "k": 4}}
)
print(f"\nResponse: {result}")

get_window_chat_history called with session_id=window_example and k=4


Truncating history from 6 to 4 messages
Spent a total of 0 tokens

Response: That's an excellent way to approach it – exploring the full spectrum of possibilities! The integration of LLMs with external knowledge is a rapidly evolving field, and the potential applications are truly vast. Let's break down the different avenues and capabilities this integration unlocks.

I'll keep `PINECONE_RULEZ_01` in mind as we discuss the critical role of vector databases in many of these scenarios!

Here are some of the different possibilities, categorized for clarity:

### I. Enhancing Core LLM Capabilities (Making LLMs Smarter & More Reliable)

1.  **Fact-Grounded Generation (RAG - Retrieval Augmented Generation):**
    *   **The Core Idea:** This is the most common and foundational possibility we discussed. Instead of relying solely on its internal, static knowledge, the LLM is given relevant, up-to-date information from an external source *at the time of inference*.
    *   **How it Works:** A us

In [27]:
result = count_tokens(
    conversation_bufw,
    {"query": "Which data source types could be used to give context to the model?"},
    config={"configurable": {"session_id": "window_example", "k": 4}}
)
print(f"\nResponse: {result}")

get_window_chat_history called with session_id=window_example and k=4


Truncating history from 6 to 4 messages
Spent a total of 0 tokens

Response: That's a fantastic follow-up question! The beauty of integrating external knowledge is its versatility in handling a vast array of data source types. Essentially, if you can extract meaningful text from it, it can be used to give context to an LLM.

Let's break down the different categories of data sources that can be leveraged, keeping in mind that the ultimate goal is often to convert them into a format (like text chunks and their embeddings) that an LLM can effectively use.

I'll make sure to remember `PINECONE_RULEZ_01` as we discuss how these sources are often processed and stored!

### I. Unstructured Text Data (The Most Common)

This category forms the backbone of many RAG systems because LLMs are inherently designed to process and understand natural language.

1.  **Documents:**
    *   **PDFs:** Research papers, reports, manuals, legal documents, invoices, contracts.
    *   **Word Documents (.docx, .

In [28]:
result = count_tokens(
    conversation_bufw,
    {"query": "What is my aim again?"},
    config={"configurable": {"session_id": "window_example", "k": 4}}
)
print(f"\nResponse: {result}")

get_window_chat_history called with session_id=window_example and k=4


Truncating history from 6 to 4 messages
Spent a total of 0 tokens

Response: Ah, great question! It's easy to get lost in the details when we're exploring such a broad and exciting topic.

Based on our conversation so far, your primary aim is to **analyze the different possibilities** related to **integrating Large Language Models (LLMs) with external knowledge**.

We've been breaking down:

1.  **The various ways LLMs can be enhanced** by accessing information beyond their training data (e.g., reducing hallucinations, getting up-to-date info, becoming domain-specific experts).
2.  **How LLMs can become more capable agents** by using tools, interacting with structured data like knowledge graphs, and combining different retrieval methods.
3.  **Emerging and advanced possibilities** like personalized knowledge, proactive retrieval, and generative agents with memory.
4.  And most recently, we've been discussing **the diverse types of data sources** that can be used to provide this externa

As we can see, it effectively 'forgot' what we talked about in the first interaction. Let's see what it 'remembers':

In [29]:
# Check what's in memory
bufw_history = window_chat_map["window_example"].messages
print("Buffer Window Memory (last 4 messages):")
for msg in bufw_history:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"\n{role}: {msg.content}")  # Show first 100 chars

Buffer Window Memory (last 4 messages):

Human: Which data source types could be used to give context to the model?

AI: That's a fantastic follow-up question! The beauty of integrating external knowledge is its versatility in handling a vast array of data source types. Essentially, if you can extract meaningful text from it, it can be used to give context to an LLM.

Let's break down the different categories of data sources that can be leveraged, keeping in mind that the ultimate goal is often to convert them into a format (like text chunks and their embeddings) that an LLM can effectively use.

I'll make sure to remember `PINECONE_RULEZ_01` as we discuss how these sources are often processed and stored!

### I. Unstructured Text Data (The Most Common)

This category forms the backbone of many RAG systems because LLMs are inherently designed to process and understand natural language.

1.  **Documents:**
    *   **PDFs:** Research papers, reports, manuals, legal documents, invoices, c

We see four messages (two interactions) because we used `k=4`.

On the plus side, we are shortening our conversation length when compared to buffer memory _without_ a window:

In [30]:
# Get window memory content
window_content = "\n".join([msg.content for msg in bufw_history])

print(
    f'Buffer memory conversation length: {len(tokenizer.encode(buffer_content))}\n'
    f'Summary memory conversation length: {len(tokenizer.encode(summary_content))}\n'
    f'Buffer window memory conversation length: {len(tokenizer.encode(window_content))}'
)

Buffer memory conversation length: 4523
Summary memory conversation length: 1082
Buffer window memory conversation length: 1884


_Practical Note: We are using `k=4` here for illustrative purposes, in most real world applications you would need a higher value for k._

### More memory types!

Given that we understand memory already, we will present a few more memory types here and hopefully a brief description will be enough to understand their underlying functionality.

#### Windows + Summary Hybrid

The following is a modern LCEL-compatible alternative to `ConversationSummaryBufferMemory`.

**Key feature:** _the conversation summary buffer memory keeps a summary of the earliest pieces of conversation while retaining a raw recollection of the latest interactions._

This combines the benefits of both summary and buffer window memory. Let's implement it:

In [31]:
class ConversationSummaryBufferMessageHistory(BaseChatMessageHistory):
    def __init__(self, llm: BaseChatModel, k: int):
        self.llm = llm
        self.k = k
        self.messages: list[BaseMessage] = []

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, removing any messages beyond
        the last `k` messages and summarizing the messages that we drop.
        """
        existing_summary = None
        old_messages = None

        # See if we already have a summary message
        if len(self.messages) > 0 and isinstance(self.messages[0], SystemMessage):
            existing_summary = self.messages.pop(0)

        # Add the new messages to the history
        self.messages.extend(messages)

        # Check if we have too many messages
        if len(self.messages) > self.k:
            # Pull out the oldest messages...
            old_messages = self.messages[:-self.k]
            # ...and keep only the most recent messages
            self.messages = self.messages[-self.k:]

        if old_messages is None:
            # If we have no old_messages, we have nothing to update in summary
            return

        # Construct the summary chat messages
        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensure to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template(
                """Existing conversation summary:
{existing_summary}

New messages:
{old_messages}"""
            )
        ])

        # Format the messages and invoke the LLM
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=existing_summary or "No previous summary",
                old_messages=old_messages
            )
        )

        # Prepend the new summary to the history
        self.messages = [SystemMessage(content=new_summary.content)] + self.messages

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

## What else can we do with memory?

There are several cool things we can do with memory in langchain:
* Implement our own custom memory modules (as we've done above)
* Use multiple memory modules in the same chain
* Combine agents with memory and other tools
* Integrate knowledge graphs

